In [0]:

# Databricks notebook source

# ============================================================
# FORMULA1 INCREMENTAL LOAD - SILVER HELPER
# ============================================================
# Purpose:
# Creates Silver Delta data and merges incremental records
# based on the primary key.
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable


def write_to_silver(
    input_df,
    target_path,
    merge_condition,
    columns_to_update
):
    """
    Creates the Silver Delta data if it does not exist.
    Otherwise, merges incremental records into existing data.
    """

    # Add audit timestamps
    final_df = (
        input_df
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp())
    )

    # First run: create Silver Delta data
    if not DeltaTable.isDeltaTable(spark, target_path):

        (
            final_df.write
            .format("delta")
            .mode("overwrite")
            .save(target_path)
        )

        print("Silver Delta data created successfully.")

    # Subsequent runs: merge incremental data
    else:

        delta_table = DeltaTable.forPath(spark, target_path)

        update_map = {
            column: f"s.`{column}`"
            for column in columns_to_update
            if column not in ["created_timestamp", "updated_timestamp"]
        }

        update_map["updated_timestamp"] = "s.updated_timestamp"

        (
            delta_table.alias("t")
            .merge(
                final_df.alias("s"),
                merge_condition
            )
            .whenMatchedUpdate(
                condition="s.batch_id >= t.batch_id",
                set=update_map
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("Silver Delta data merged successfully.")
